In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in {'Baseline_Training', 'CTC_models', 'PhoneticFeatures_Training'}:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project_paths import (
    BASELINE_MODELS_DIR,
    CTC_MODELS_DIR,
    DEFAULT_BASELINE_MODEL,
    DEFAULT_CTC_ATTENTION_MODEL,
    DEFAULT_CTC_MODEL,
    DEFAULT_PHONETIC_MODEL_2CLASS,
    DEFAULT_PHONETIC_MODEL_3CLASS,
    EXAMPLE_EMBEDDINGS,
    EXAMPLE_PHONEMES,
    EXAMPLE_SEG_B2,
    EXAMPLE_SEG_B4,
    EXAMPLE_WAV,
    PHONEME_CHOICES_SUMMARY,
    PHONETIC_MODELS_DIR,
    TABLES_DIR,
)


# Phonetic Feature Inference

Ноутбук для пофреймового инференса обученной модели: загрузка набора эмбеддингов, предсказание признаков для каждого окна и декодирование в IPA-кандидаты.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from phonetic_inference import (
    build_lookup_features,
    decode_feature_classes_sequence,
    find_matching_ipa_for_sequence,
    load_checkpoint,
    load_checkpoint_model,
    predict_feature_classes_sequence,
)


## Config

Задай пути к чекпоинту, IPA-таблице и `.npy` файлу с эмбеддингами. Ожидается тензор формы `[num_frames, input_dim]`.


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

example_path = Path(r'C:\Users\ext-ananeva\phon_clust\gitlab\ssl_phoneme_clusterizer\Main_project\examples')
checkpoint_path = str(DEFAULT_PHONETIC_MODEL_2CLASS)

ipa_all_path  = r'C:\Users\ext-ananeva\venv3.7\Lib\site-packages\panphon\data\ipa_all.csv' 
ipa_rus_path  = r'C:\Users\ext-ananeva\venv3.7\Lib\site-packages\panphon\data\ipa_rus.csv' 
ipa_reduced_path  = r'C:\Users\ext-ananeva\venv3.7\Lib\site-packages\panphon\data\ipa_reduced.csv' 
ipa_selected_path = r"C:\Users\ext-ananeva\venv3.7\Lib\site-packages\panphon\data\ipa_selected.csv"
embs_file = EXAMPLE_EMBEDDINGS
label_file = EXAMPLE_PHONEMES
wav_file = EXAMPLE_WAV

get_embbs = True

wav_file =  r'C:\Users\ext-ananeva\Downloads\SA1.WAV.wav'

extra_features = {
    'hitone': 0,
    'hireg': 0,
    'sg':-1,
    'cg':-1,
    'tense':-1,
    'long':-1,
    'velaric':-1
}

overrides = {
    # 'hi': -1,
}




In [ ]:
df_all = pd.read_csv(ipa_rus_path, index_col=0)

mapping = {"-": -1, "+": 1, "0": 0}
df_all = df_all.replace(mapping)

In [ ]:
# df_all = df_all[df_all.index !='ə']
# df_all = df_all[df_all.index != 'ɻʲ']

In [ ]:
df_all.index[1]

In [ ]:
list_of_features = df_all.columns.to_list()
list_of_features

In [ ]:
checkpoint = load_checkpoint(checkpoint_path, device)
model, checkpoint, model_config = load_checkpoint_model(
    checkpoint_path=checkpoint_path,
    device=device,
)

model_config


## Load Embeddings

Ниже загрузка ровно в том формате, который ты показала: `numpy -> torch -> transpose`.


In [ ]:
import torchaudio
from torchaudio import functional as FA

from content_manager.vencoder.HubertSoft import HubertSoft
if get_embbs:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    content_encoder = HubertSoft(device=device)
    wave, sr = torchaudio.load(wav_file)
    res_wave = FA.resample(wave, sr, 16000)
    
    ext_emb = content_encoder.encoder(res_wave[0].to(device))
    normed_emb = ext_emb[0] / (ext_emb[0] ** 2).sum(0, keepdims=True) ** 0.5
    normed_emb = normed_emb.cpu().numpy()
    
    np.save(EXAMPLE_EMBEDDINGS, normed_emb)
    normed_emb = np.load(EXAMPLE_EMBEDDINGS)
    normed_emb = torch.tensor(normed_emb, dtype=torch.float32)
    normed_emb = normed_emb.T
    
else:
    normed_emb = np.load(embs_file)
    normed_emb = torch.tensor(normed_emb, dtype=torch.float32)
    normed_emb = normed_emb

normed_emb.shape

## Predict Feature Classes Per Frame

Каждая строка результата соответствует одному окну аудио.


In [ ]:
predicted_classes_df = predict_feature_classes_sequence(
    model,
    normed_emb.to(device),
    model_config['active_heads'],
)

predicted_classes_df.head()


In [ ]:
predicted_features_df = decode_feature_classes_sequence(predicted_classes_df)
predicted_features_df.head()


In [ ]:
list_of_features

In [ ]:
predicted_features_df = predicted_features_df.assign(**extra_features)[list_of_features]


## IPA Lookup Per Frame

Для каждого окна собирается `find_features`, после чего ищутся совпадения в IPA-таблице.


In [ ]:
def find_matching_ipa_for_sequence_test(df_all, predicted_features_df):

    rows = []

    for frame_idx, feature_row in predicted_features_df.iterrows():
        feature_row= feature_row.to_dict()
        
        mask = (df_all[list(feature_row)] == pd.Series(feature_row)).all(axis=1)

        result = df_all[mask]
        if len(result.index)!=0:
            ipa_symb = result.index.to_list()
        else:
            ipa_symb = None

        rows.append(
            {
                "frame_idx": frame_idx,
                "lookup_features": feature_row,
                "ipa_matches": ipa_symb,
                'num_matches': len(result.index.to_list()),

            }
        )

    return pd.DataFrame(rows)



In [ ]:
df_sample = df_all#df_all.sample(n=100)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

X = df_sample.values
y = df_sample.index

rf = RandomForestClassifier()
rf.fit(X, y)

weights = rf.feature_importances_

In [ ]:
df_all.index

In [ ]:
weights_dict = {}

for i, feature in enumerate(list(df_all.columns)):
    weights_dict[feature] = weights[i]
    
weights_dict

In [ ]:
weights_dict['round'] = weights_dict['round']*2
weights_dict['lat'] = weights_dict['lat']*4

In [ ]:
weights_dict_norm = {}
m = np.sum(list(weights_dict.values()))
std = np.std(list(weights_dict.values()))
for i, feature in enumerate(list(weights_dict.keys())):
    weights_dict_norm[feature] = (weights_dict[feature])/m

In [ ]:
sum(list(weights_dict_norm.values()))

In [ ]:
weights_dict = {}

for i, feature in enumerate(list(df_all.columns)):
    weights_dict[feature] = 1
    
weights_dict

In [ ]:
# Просто расстояние Хэмминга - считаем сколько признаков в целевом и таргетном не совпадают
def find_closest_ipa_by_distanse(df_all, predicted_features_df):

    rows = []

    for frame_idx, feature_row in predicted_features_df.iterrows():
        feature_row = feature_row.to_dict()

        distances = (df_all[list(feature_row)] != pd.Series(feature_row)).sum(axis=1)

        min_dist = distances.min()

        result = df_all[distances == min_dist]

        rows.append({
            "frame_idx": frame_idx,
            "lookup_features": feature_row,
            "ipa_matches": result.index.to_list(),
            "num_matches": len(result.index.to_list()),
            "distance": min_dist
        })

    return pd.DataFrame(rows)

# Расстояние Хэмминга но с добалением словарей с весами
def find_closest_ipa_with_weights(df_all, predicted_features_df, weights):

    rows = []

    for frame_idx, feature_row in predicted_features_df.iterrows():
        feature_row = feature_row.to_dict()

        features = list(feature_row)

        # вектор весов
        w = pd.Series(weights)[features]

        # матрица несовпадений
        diff = (df_all[features] != pd.Series(feature_row)).astype(int)

        # взвешенное расстояние
        distances = (diff * (w*10)).sum(axis=1)

        min_dist = distances.min()

        result = df_all[distances == min_dist]

        rows.append({
            "frame_idx": frame_idx,
            "lookup_features": feature_row,
            "ipa_matches": result.index.to_list(),
            "num_matches": len(result.index.to_list()),
            "distance": min_dist
        })

    return pd.DataFrame(rows)

# веса полученные через feature_importance в RandomForest
random_forest_weights = {'syl': 0.03638454252335947,
                         'son': 0.03142632784111691,
                         'cons': 0.0161493332674742,
                         'cont': 0.04929991055401256,
                         'delrel': 0.0570660593305012,
                         'lat': 0.027176941156326375,
                         'nas': 0.024583428106599022,
                         'strid': 0.03428711557688485,
                         'voi': 0.06693509977117414,
                         'sg': 0.051739540475226733,
                         'cg': 0.05608930405671631,
                         'ant': 0.0618932274391491,
                         'cor': 0.02823026651275813,
                         'distr': 0.054318167647589934,
                         'lab': 0.034126128693469766,
                         'hi': 0.07714964209925472,
                         'lo': 0.07934820216067466,
                         'back': 0.042952076549207546,
                         'round': 0.07194215463397328,
                         'velaric': 0.01295649187693806,
                         'tense': 0.03204794328603258,
                         'long': 0.05389809644156048,
                         'hitone': 0.0,
                         'hireg': 0.0}

from sklearn.neighbors import NearestNeighbors

def find_closest_ipa_with_nearestneighbours(df_all, predicted_features_df, k=3):
    X = df_all.values
    
    nn = NearestNeighbors(n_neighbors=k, metric='hamming')
    nn.fit(X)
    
    rows = []
    
    for frame_idx, feature_row in predicted_features_df.iterrows():
        x = feature_row.values.reshape(1, -1)
        distances, indices = nn.kneighbors(x)
        
        ipa_matches = df_all.index[indices[0]].to_list()
        rows.append({
            'frame_idx': frame_idx,
            'lookup_features': feature_row,
            'ipa_matches': ipa_matches,
            "num_matches": len(result.index.to_list()),
            'distance': distances[0][0] if k == 1 else distances[0].tolist(),
        })
        
    return pd.DataFrame(rows)
        


In [ ]:
ipa_matches_df = find_closest_ipa_with_weights(   
    df_all=df_all,
    predicted_features_df=predicted_features_df,
    weights=random_forest_weights

)

ipa_matches_df.head()


In [ ]:
ipa_matches_df

In [ ]:
result_transc = []
for frame_idx, row in ipa_matches_df.iterrows():
    if row['ipa_matches'] is not None:
        if '̈' in row['ipa_matches'][0] and len(row['ipa_matches'])>1:
            result_transc.append(row['ipa_matches'][1])
        else:
            result_transc.append(row['ipa_matches'][0])
    else:
        result_transc.append(' ')
    
'_'.join(result_transc) 

In [ ]:
full_result_df = pd.concat(
    [
        predicted_classes_df.add_prefix('class_'),
        predicted_features_df.add_prefix('feat_'),
        ipa_matches_df[['ipa_matches', 'num_matches']],
    ],
    axis=1,
)

full_result_df.head()


## Inspect One Frame

Можно выбрать конкретный frame и посмотреть, какие признаки и IPA-кандидаты там получились.


In [ ]:
frame_idx = 0

frame_classes = predicted_classes_df.iloc[frame_idx].to_dict()
frame_features = predicted_features_df.iloc[frame_idx].to_dict()
frame_lookup = build_lookup_features(frame_features, extra_features=extra_features, overrides=overrides)
frame_matches = ipa_matches_df.iloc[frame_idx]

frame_classes, frame_features, frame_lookup, frame_matches


## Manual Lookup For One Frame

Если нужно руками исправить признаки для конкретного окна и повторить поиск.


In [ ]:
df_all = pd.read_csv(ipa_all_path, index_col=0)
find_features = dict(frame_lookup)

# пример ручной правки
# find_features['hi'] = -1
# find_features['tense'] = 0

mask = (df_all[list(find_features)] == pd.Series(find_features)).all(axis=1)
result = df_all[mask]
result


## Save Results

При необходимости можно сохранить пофреймовые результаты в CSV.


In [ ]:
# full_result_df.to_csv('ata1101_framewise_predictions.csv', index=True)
